# 1. Objective
  The Objective of the notebook is to ingest different external data sources into Snowflake DB & to create derived tables out of raw data so that the data can be tracked and used in the downstream processes.

  Following are the different data sources which we are pulling into Snowflake DB
  - Regional CPI
  - Regional Unemployment Rate
  - HPI
  - Holiday data

  Following are the different tables that are created out of raw data
  - Week day weights for borderline week adjustment
  

# 2. Imports

In [ ]:
# Import python packages
import streamlit as st
import pandas as pd
import holidays

from snowflake.snowpark.window import Window
from snowflake.snowpark import functions as F
from snowflake.snowpark import types as T
from snowflake.snowpark import Session

# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()

In [ ]:
print("✅ Snowpark Session Initialized Successfully!")
print("Current Database:", session.get_current_database())
print("Current Schema:", session.get_current_schema())

# 3. Setup environment

## 3.1. Load Config

In [ ]:
import yaml
with open("config_new_PROD.yaml") as file:
    app_config = yaml.safe_load(file)

## 3.2. Update Output Database, Schema , table

In [ ]:
output_database = app_config["general_inputs"]["output_database"]
output_schema = app_config["general_inputs"]["output_schema"]
print(output_database, output_schema)

In [ ]:
session.use_database(output_database)
session.use_schema(output_schema)
unemp_cpi_output_table_name = "PROD_UNEMP_CPI_DATA"
holiday_output_table_name = "PROD_US_HOLIDAY_DATA"
hpi_output_table_name = "PROD_HPI_DATA"
day_imp_output_table_name = "PROD_DAY_IMPORTANCE"

In [ ]:
# Example check (optional)
print("✅ Snowpark Session Initialized Successfully!")
print("Current Database:", session.get_current_database())
print("Current Schema:", session.get_current_schema())

# 4. Utilities

## 4.1. Export data API

In [ ]:
def snowpark_type_to_sql(datatype):
    if isinstance(datatype, T.StringType):
        return "VARCHAR"
    if isinstance(datatype, T.IntegerType):
        return "INTEGER"
    if isinstance(datatype, T.LongType):
        return "BIGINT"
    if isinstance(datatype, T.ShortType):
        return "SMALLINT"
    if isinstance(datatype, T.ByteType):
        return "BYTEINT"
    if isinstance(datatype, T.BooleanType):
        return "BOOLEAN"
    if isinstance(datatype, T.FloatType):
        return "FLOAT"
    if isinstance(datatype, T.DoubleType):
        return "DOUBLE"
    if isinstance(datatype, T.DecimalType):
        return f"NUMBER({datatype.precision},{datatype.scale})"
    if isinstance(datatype, T.DateType):
        return "DATE"
    if isinstance(datatype, T.TimestampType):
        return "TIMESTAMP_NTZ"
    if isinstance(datatype, T.TimestampTimeZoneType):
        return "TIMESTAMP_TZ"
    if isinstance(datatype, T.VariantType):
        return "VARIANT"

    # Fallback (safe but explicit)
    return "VARCHAR"

In [ ]:
def evolve_schema_and_append_with_TS(
    session: Session,
    src_df: str,
    target_table_name: str,
):
    """
    Schema-drift tolerant append:
      - Adds new columns from source to target
      - Appends data with column alignment
    """

    #src_df = session.table(source_table)
    tgt_df = session.table(target_table_name)
    src_df = src_df.with_column("LOAD_TS", F.current_timestamp())
    src_df = src_df.with_column("EXECUTION_YEAR_MONTH",F.date_format(F.current_date(), "YYYY-MM"))
    src_row_count = src_df.count()
    src_col_count = len(src_df.columns)

    print("Source DF shape ->",src_row_count,",",src_col_count)

    tgt_row_count = tgt_df.count()
    tgt_col_count = len(tgt_df.columns)

    print("Target table shape ->",tgt_row_count,",",tgt_col_count)

    src_schema = {f.name.upper(): f.datatype for f in src_df.schema.fields}
    tgt_schema = {f.name.upper(): f.datatype for f in tgt_df.schema.fields}

    # 1️⃣ Add missing columns to target
    new_columns = src_schema.keys() - tgt_schema.keys()
    if len(new_columns)>0:
        print("New Columns in Source dataframe found. Target table will be altered")
    else:
        print("No New columns found in source dataframe")
    for col_name in new_columns:
        datatype = snowpark_type_to_sql(src_schema[col_name])
        print(col_name)
        print(datatype)
        ddl = f"""
            ALTER TABLE {target_table_name}
            ADD COLUMN IF NOT EXISTS "{col_name}" {datatype}
        """
        print(ddl)
        ddl_op = session.sql(ddl)
        print(ddl_op.collect())
    
    # Refresh target after DDL
    tgt_df = session.table(target_table_name)

    # Align columns for insert
    common_cols = [
        F.col(c)
        for c in tgt_df.schema.names
        if c.upper() in src_schema
    ]
    
    (
        src_df
        .select(common_cols)
        .write
        .mode("append")
        .save_as_table(target_table_name)
    )
    print("Appended ",src_row_count," rows of data to Target table with latest TS successfully")

# 5. External Data Processing

## 5.1. Unemplyment Rate

### 5.1.1. MI-01

In [ ]:
import requests
import json
import pandas as pd
headers = {'Content-type': 'application/json'}
data = json.dumps({"seriesid": ['LASST260000000000003'],"startyear":"2022", "endyear":"2025"})
p = requests.post('https://api.bls.gov/publicAPI/v2/timeseries/data/', data=data, headers=headers)
json_data = json.loads(p.text)
df = pd.DataFrame()
rows = []
for series in json_data['Results']['series']:
    seriesId = series['seriesID']
    for item in series['data']:
        year = item['year']
        period = item['period']
        value = item['value']
        footnotes=""
        for footnote in item['footnotes']:
            if footnote:
                footnotes = footnotes + footnote['text'] + ','
        if 'M01' <= period <= 'M12':
            rows.append([seriesId,year,period,value,footnotes[0:-1]])

In [ ]:
michigan_df = pd.DataFrame(rows, columns=["SERIESID","YEAR","MONTH","UNEMPLOYMENT_RATE","FOOTNOTES"])
michigan_df["REGIONNAME"] = "MI-01"
michigan_df

### 5.1.2. NY-01

In [ ]:
import requests
import json
import pandas as pd
headers = {'Content-type': 'application/json'}
data = json.dumps({"seriesid": ['LASST360000000000003'],"startyear":"2022", "endyear":"2025"})
p = requests.post('https://api.bls.gov/publicAPI/v2/timeseries/data/', data=data, headers=headers)
json_data = json.loads(p.text)
df = pd.DataFrame()
rows = []
for series in json_data['Results']['series']:
    seriesId = series['seriesID']
    for item in series['data']:
        year = item['year']
        period = item['period']
        value = item['value']
        footnotes=""
        for footnote in item['footnotes']:
            if footnote:
                footnotes = footnotes + footnote['text'] + ','
        if 'M01' <= period <= 'M12':
            rows.append([seriesId,year,period,value,footnotes[0:-1]])

In [ ]:
ny_df = pd.DataFrame(rows, columns=["SERIESID","YEAR","MONTH","UNEMPLOYMENT_RATE","FOOTNOTES"])
ny_df["REGIONNAME"] = "NY-01"
ny_df

In [ ]:
all_df_unemp = pd.concat([michigan_df, ny_df], axis = 0)
all_df_unemp

## 5.2. CPI

### 5.2.1. MI-01

In [ ]:
import requests
import json
import pandas as pd
headers = {'Content-type': 'application/json'}
data = json.dumps({"seriesid": ['CUUR0200SA0'],"startyear":"2022", "endyear":"2025"})
p = requests.post('https://api.bls.gov/publicAPI/v2/timeseries/data/', data=data, headers=headers)
json_data = json.loads(p.text)
df = pd.DataFrame()
rows = []
for series in json_data['Results']['series']:
    seriesId = series['seriesID']
    for item in series['data']:
        year = item['year']
        period = item['period']
        value = item['value']
        footnotes=""
        for footnote in item['footnotes']:
            if footnote:
                footnotes = footnotes + footnote['text'] + ','
        if 'M01' <= period <= 'M12':
            rows.append([seriesId,year,period,value,footnotes[0:-1]])

In [ ]:
michigan_df = pd.DataFrame(rows, columns=["SERIESID","YEAR","MONTH","CPI","FOOTNOTES"])
michigan_df["REGIONNAME"] = "MI-01"
michigan_df

### 5.2.2. NY-01

In [ ]:
import requests
import json
import pandas as pd
headers = {'Content-type': 'application/json'}
data = json.dumps({"seriesid": ['CUUR0100SA0'],"startyear":"2022", "endyear":"2025"})
p = requests.post('https://api.bls.gov/publicAPI/v2/timeseries/data/', data=data, headers=headers)
json_data = json.loads(p.text)
df = pd.DataFrame()
rows = []
for series in json_data['Results']['series']:
    seriesId = series['seriesID']
    for item in series['data']:
        year = item['year']
        period = item['period']
        value = item['value']
        footnotes=""
        for footnote in item['footnotes']:
            if footnote:
                footnotes = footnotes + footnote['text'] + ','
        if 'M01' <= period <= 'M12':
            rows.append([seriesId,year,period,value,footnotes[0:-1]])

In [ ]:
ny_df = pd.DataFrame(rows, columns=["SERIESID","YEAR","MONTH","CPI","FOOTNOTES"])
ny_df["REGIONNAME"] = "NY-01"
ny_df

In [ ]:
all_df_cpi = pd.concat([michigan_df, ny_df], axis = 0)
all_df_cpi

In [ ]:
all_df_cpi.dtypes

In [ ]:
cpi_data = all_df_cpi.copy()

In [ ]:
unemp_data = all_df_unemp.copy()

In [ ]:
# ---------------------------------------
# 1. Create YEAR_MONTH safely
# ---------------------------------------
unemp_data["YEAR_MONTH"] = (
    unemp_data["YEAR"].astype(str) + "-" +
    unemp_data["MONTH"].astype(str).str.replace("M", "")
)

cpi_data["YEAR_MONTH"] = (
    cpi_data["YEAR"].astype(str) + "-" +
    cpi_data["MONTH"].astype(str).str.replace("M", "")
)

# ---------------------------------------
# 2. Keep ONLY required columns
# ---------------------------------------
unemp_df = unemp_data[
    ["REGIONNAME", "YEAR_MONTH", "UNEMPLOYMENT_RATE"]
]

cpi_df = cpi_data[
    ["REGIONNAME", "YEAR_MONTH", "CPI"]
]

# ---------------------------------------
# 3. Merge cleanly (no _x, _y)
# ---------------------------------------
final_mac_data = unemp_df.merge(
    cpi_df,
    on=["REGIONNAME", "YEAR_MONTH"],
    how="right"
)

# ---------------------------------------
# 4. Rename columns
# ---------------------------------------
final_mac_data = final_mac_data.rename(
    columns={
        "UNEMPLOYMENT_RATE": "REGIONAL_UNEMPLOYMENT_RATE",
        "CPI": "REGIONAL_CPI"
    }
)

final_mac_data["REGIONAL_UNEMPLOYMENT_RATE"] = pd.to_numeric(final_mac_data["REGIONAL_UNEMPLOYMENT_RATE"], errors="coerce")
final_mac_data["REGIONAL_CPI"] = pd.to_numeric(final_mac_data["REGIONAL_CPI"], errors="coerce")

final_mac_data

In [ ]:
final_mac_data.dtypes

In [ ]:
final_mac_data_spk = session.create_dataframe(final_mac_data)

In [ ]:
evolve_schema_and_append_with_TS(session, src_df = final_mac_data_spk, target_table_name = unemp_cpi_output_table_name)

## 5.3. Holiday data - US

In [ ]:
min_year = int(app_config["data_harmonization_inputs"]["min_year"])
max_year = int(app_config["data_harmonization_inputs"]["max_year"])
print("Minimum year to pull data for --->", min_year)
print("Maximum year to pull data for --->", max_year)

In [ ]:
us_holidays = holidays.UnitedStates(years=range(min_year, max_year+1))

df_holidays = pd.DataFrame(
    [(date, name) for date, name in us_holidays.items()],
    columns=["DATE", "HOLIDAY"]
)

df_holidays["DATE"] = pd.to_datetime(df_holidays["DATE"])

daily_dates = pd.date_range(start="01-01-"+str(min_year), end="31-12-"+str(max_year), freq="D", inclusive="both")
daily_dates_df = pd.DataFrame({"DATE":daily_dates})
     

In [ ]:
holiday_data_by_date = daily_dates_df.merge(df_holidays, on = "DATE", how = "left")

In [ ]:
holiday_data_by_date

In [ ]:
holiday_flags = (
    holiday_data_by_date.assign(flag=1)
      .pivot_table(index="DATE", columns="HOLIDAY", values="flag", fill_value=0)
      .reset_index()
)
holiday_flags

In [ ]:
holiday_flags.rename(columns = {"DATE":"JOIN_DATE"}, inplace = True)
holiday_flags.columns = (
    holiday_flags.columns
    .str.upper()                           # Capitalize (full uppercase)
    .str.replace(r"[^A-Z0-9]+", "_", regex=True)   # Replace special chars with _
    .str.strip("_")                        # Remove leading/trailing underscores
)
holiday_cols = [i for i in holiday_flags.columns if i != "JOIN_DATE"]
holiday_cols

In [ ]:
holiday_flags

In [ ]:
# ---------------------------------------
# 2. Create Snowpark DataFrame (ONLY ONCE)
# ---------------------------------------
holiday_flags_spk = session.create_dataframe(holiday_flags)

In [ ]:
holiday_flags_spk.count()

In [ ]:
holiday_flags_spk

In [ ]:
evolve_schema_and_append_with_TS(session, src_df = holiday_flags_spk, target_table_name = holiday_output_table_name)

## 5.4. HPI

In [ ]:
# ---------------------------------------
# 3. Run SQL & Store as Snowpark DataFrame
# ---------------------------------------
df_hpi = session.sql("""
SELECT 
    CASE
        WHEN variable ILIKE '%traditional%' THEN 'traditional'
        WHEN variable ILIKE '%non-metro%' THEN 'non-metro'
        ELSE 'others'
    END AS hpi_type,

    CASE 
        WHEN variable ILIKE '%all-transactions%' THEN 'all-transactions'
        WHEN variable ILIKE '%expanded-data%' THEN 'expanded-data'
        WHEN variable ILIKE '%purchase-only%' THEN 'purchase-only'
        ELSE 'others'
    END AS hpi_flavor,

    CASE 
        WHEN variable ILIKE '%quarterly%' THEN 'quarterly'
        WHEN variable ILIKE '%monthly%' THEN 'monthly'
        ELSE 'others'
    END AS frequency,

    level,
    geo_name AS place_name,
    YEAR(date) AS year,

    CASE 
        WHEN MONTH(date) = 3  THEN 1
        WHEN MONTH(date) = 6  THEN 2
        WHEN MONTH(date) = 9  THEN 3
        WHEN MONTH(date) = 12 THEN 4
    END AS period,

    value AS index_nsa

FROM SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.FHFA_HOUSE_PRICE_TIMESERIES HPI
JOIN SNOWFLAKE_PUBLIC_DATA_FREE.PUBLIC_DATA_FREE.GEOGRAPHY_INDEX Geo
    ON HPI.GEO_ID = Geo.GEO_ID

WHERE level = 'State'
  AND geo_name IN ('New York', 'Michigan')
  AND variable ILIKE '%nsa%'
  AND variable = 'FHFA_HPI_traditional_all-transactions_quarterly_NSA'
""")


# ---------------------------------------
# 4. Create REGIONNAME Column
# ---------------------------------------
df_hpi_with_region = df_hpi.with_column(
    "REGIONNAME",
    F.when(F.col("PLACE_NAME") == "Michigan", "MI-01")
    .when(F.col("PLACE_NAME") == "New York", "NY-01")
    .otherwise("UNKNOWN")
)

# ---------------------------------------
# 5. Create YEAR_QUARTER Column (Snowpark way)
# ---------------------------------------
hpi_data = df_hpi_with_region.with_column(
    "YEAR_QUARTER",
    F.concat(
        F.col("YEAR").cast("string"),
        F.lit("-Q"),
        F.col("PERIOD").cast("string")
    )
)

# ---------------------------------------
# 6. Rename INDEX_NSA → HPI
# ---------------------------------------
hpi_data = hpi_data.with_column_renamed("INDEX_NSA", "HPI")

# ---------------------------------------
# 7. Select Final Columns
# ---------------------------------------
hpi_data = hpi_data.select(
    "YEAR_QUARTER",
    "REGIONNAME",
    "HPI"
)


In [ ]:
hpi_data

In [ ]:
evolve_schema_and_append_with_TS(session, src_df = hpi_data, target_table_name = hpi_output_table_name)

# 6. Internal Data Processing
  - Weights for each weekday is calculated from Jan 2022 till current date for adjusting the borderline week forecasts later in the pipeline

## 6.1. Day Importance

In [ ]:
J = session.table("orange_zone_sbx_ta.member.joins").alias("J")
D = session.table("orange_zone_sbx_ta.reporting.dim_date").alias("D")
S = session.table("orange_zone_sbx_ta.studio.dim_studio").alias("S")

In [ ]:
df_day_weights = (
    J
    .join(D, D.DATE == J.JOIN_DATE)
    .join(S, J.MBO_STUDIO_ID == S.SOURCE_MBOSTUDIOID)
    .filter(
        (F.col("shiptocountry") == "United States") &
        (F.col("JOIN_DATE") >= "2022-07-01") &
        (F.col("JOIN_DATE") <= F.current_date()) &
        (F.col("is_network_new_join") == True)
    )
)

In [ ]:
df_day_weights

In [ ]:
df_day_weights_grouped = (
    df_day_weights
    .group_by("day_of_week", "dayname")
    .agg(
        F.count("*").alias("cnt")
    )
)

In [ ]:
df_day_weights_grouped

In [ ]:
total_window = Window.partition_by()

result = (
    df_day_weights_grouped
    .with_column(
        "weights",
        F.col("cnt") / F.sum(F.col("cnt")).over(total_window)
    )
    .order_by("day_of_week")
)

In [ ]:
result = result.drop(["DAY_OF_WEEK"])

In [ ]:
result = result.with_column("DAY_OF_WEEK",F.when(F.col("DAYNAME") == "Monday",F.lit(0))
                                 .when(F.col("DAYNAME") == "Tuesday",F.lit(1))
                                 .when(F.col("DAYNAME") == "Wednesday",F.lit(2))
                                 .when(F.col("DAYNAME") == "Thursday",F.lit(3))
                                 .when(F.col("DAYNAME") == "Friday",F.lit(4))
                                 .when(F.col("DAYNAME") == "Saturday",F.lit(5))
                                 .otherwise(F.lit(6))
                                 )

In [ ]:
result

In [ ]:
evolve_schema_and_append_with_TS(session, src_df = result, target_table_name = day_imp_output_table_name)